# 02 — Data Quality
Online Retail II

Audit source overlap, duplicated records, invoice identity, and transaction quality. Source rows are preserved.

## 1. Load and identify records

Load both worksheets, retain their original row positions, and confirm record counts.


In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)
source_columns = list(next(iter(sheets.values())).columns)

raw = pd.concat(
    [sheet.assign(source_sheet=name, source_row=np.arange(len(sheet)))
     for name, sheet in sheets.items()],
    ignore_index=True,
)

df = raw.copy()
df["invoice_id"] = df["Invoice"].astype("string").str.strip()
df["stock_code"] = df["StockCode"].astype("string").str.strip()
df["customer_id"] = df["Customer ID"].astype("Int64").astype("string")
df["invoice_timestamp"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["line_amount"] = df["Quantity"] * df["Price"]
df["source_record_key"] = df["source_sheet"] + ":" + df["source_row"].astype(str)

assert df["source_record_key"].is_unique
assert len(df) == sum(len(sheet) for sheet in sheets.values())
display(df.groupby("source_sheet").size().rename("rows").to_frame())
print("Total source rows:", len(df))

,rows
source_sheet,
Year 2009-2010,525461
Year 2010-2011,541910


Total source rows: 1067371


**Finding.** The two worksheets contain 525,461 and 541,910 invoice lines, respectively (1,067,371 combined). Each source row retains a worksheet name and row position.

## 2. Worksheet overlap and duplicate records

Identify identical rows within and across worksheets, and quantify overlapping invoice IDs.


In [2]:
# Compare source-column values, excluding the worksheet provenance columns.
# Hash matches are duplicate *candidates*; no rows are removed here.
df["row_fingerprint"] = pd.util.hash_pandas_object(
    df[source_columns], index=False
)
df["within_sheet_duplicate"] = df.duplicated(
    ["source_sheet", "row_fingerprint"], keep="first"
)

# Pair the first, second, etc. occurrence of each identical row across sheets.
df["fingerprint_occurrence"] = df.groupby(
    ["source_sheet", "row_fingerprint"], sort=False
).cumcount()
df["cross_sheet_duplicate_candidate"] = df.duplicated(
    ["row_fingerprint", "fingerprint_occurrence"], keep=False
)

invoice_sheets = df.groupby("invoice_id")["source_sheet"].nunique()
overlap_invoice_ids = invoice_sheets.index[invoice_sheets.gt(1)]

checks = pd.Series({
    "Source lines": len(df),
    "Within-sheet repeated lines beyond first": int(df["within_sheet_duplicate"].sum()),
    "Cross-sheet exact-row candidate pairs": int(df["cross_sheet_duplicate_candidate"].sum() // 2),
    "Invoice IDs appearing in both sheets": len(overlap_invoice_ids),
    "Rows linked to shared invoice IDs": int(df["invoice_id"].isin(overlap_invoice_ids).sum()),
})
display(checks.to_frame("count"))

display(df.loc[
    df["cross_sheet_duplicate_candidate"],
    ["source_sheet", "Invoice", "StockCode", "InvoiceDate", "Customer ID", "Quantity", "Price"]
].head(12))

,count
Source lines,1067371
Within-sheet repeated lines beyond first,12133
Cross-sheet exact-row candidate pairs,22523
Invoice IDs appearing in both sheets,1088
Rows linked to shared invoice IDs,45046


,source_sheet,Invoice,StockCode,InvoiceDate,Customer ID,Quantity,Price
502938,Year 2009-2010,536365,85123A,2010-12-01 08:26:00,17850.0,6,2.55
502939,Year 2009-2010,536365,71053,2010-12-01 08:26:00,17850.0,6,3.39
502940,Year 2009-2010,536365,84406B,2010-12-01 08:26:00,17850.0,8,2.75
502941,Year 2009-2010,536365,84029G,2010-12-01 08:26:00,17850.0,6,3.39
502942,Year 2009-2010,536365,84029E,2010-12-01 08:26:00,17850.0,6,3.39
502943,Year 2009-2010,536365,22752,2010-12-01 08:26:00,17850.0,2,7.65
502944,Year 2009-2010,536365,21730,2010-12-01 08:26:00,17850.0,6,4.25
502945,Year 2009-2010,536366,22633,2010-12-01 08:28:00,17850.0,6,1.85
502946,Year 2009-2010,536366,22632,2010-12-01 08:28:00,17850.0,6,1.85
502947,Year 2009-2010,536368,22960,2010-12-01 08:34:00,13047.0,6,4.25


**Finding.** There are 12,133 repeated lines beyond the first occurrence within worksheets and 22,523 identical-row candidate pairs across worksheets. A total of 1,088 invoice IDs occur in both worksheets, spanning 45,046 source lines. These are separate overlap measures; no records have been removed.

## 3. Invoice identity checks

Check whether invoice IDs shared by the worksheets have conflicting transaction details.


In [3]:
invoice_check = df.groupby("invoice_id", dropna=False).agg(
    source_sheets=("source_sheet", "nunique"),
    distinct_timestamps=("invoice_timestamp", "nunique"),
    distinct_customers=("customer_id", "nunique"),
    distinct_countries=("Country", "nunique"),
    first_timestamp=("invoice_timestamp", "min"),
    last_timestamp=("invoice_timestamp", "max"),
    line_count=("Invoice", "size"),
)
shared = invoice_check.loc[invoice_check["source_sheets"].gt(1)]
conflicts = shared.loc[
    shared["distinct_timestamps"].gt(1)
    | shared["distinct_customers"].gt(1)
    | shared["distinct_countries"].gt(1)
]

invoice_results = pd.Series({
    "Total distinct invoice IDs": len(invoice_check),
    "Shared invoice IDs": len(shared),
    "Shared invoice IDs with date/customer/country differences": len(conflicts),
    "All invoice IDs with multiple customer IDs": int(invoice_check["distinct_customers"].gt(1).sum()),
})
display(invoice_results.to_frame("count"))
display(conflicts.head(15))

,count
Total distinct invoice IDs,53628
Shared invoice IDs,1088
Shared invoice IDs with date/customer/country differences,1
All invoice IDs with multiple customer IDs,0


,source_sheets,distinct_timestamps,distinct_customers,distinct_countries,first_timestamp,last_timestamp,line_count
invoice_id,,,,,,,
536591,2,2,1,1,2010-12-01 16:57:00,2010-12-01 16:58:00,80


**Finding.** Among 53,628 distinct invoice IDs, 1,088 appear in both worksheets. Only invoice `536591` has differing timestamps in the shared-ID check (16:57 and 16:58 on 1 December 2010); no invoice ID maps to multiple customer IDs. The one-minute difference needs inspection before finalizing transaction keys.

## 4. Transaction quality flags

Flag missing values, cancellations, unusual stock codes, and candidate sales without removing rows.


In [4]:
df["is_cancellation_code"] = df["invoice_id"].str.upper().str.startswith("C", na=False)
df["negative_quantity"] = df["Quantity"].lt(0)
df["zero_quantity"] = df["Quantity"].eq(0)
df["nonpositive_price"] = df["Price"].le(0)
df["missing_customer"] = df["customer_id"].isna()
df["missing_description"] = df["Description"].isna()
df["invalid_timestamp"] = df["invoice_timestamp"].isna()
# Flag uncommon stock-code formats for review; they are not automatically invalid products.
df["unusual_stock_code"] = ~df["stock_code"].str.fullmatch(r"[0-9]{5}[A-Z]?", na=False)

df["positive_sale_candidate"] = (
    ~df["is_cancellation_code"]
    & df["Quantity"].gt(0)
    & df["Price"].gt(0)
    & ~df["invalid_timestamp"]
)
df["customer_sale_candidate"] = df["positive_sale_candidate"] & ~df["missing_customer"]

flags = [
    "cross_sheet_duplicate_candidate", "within_sheet_duplicate",
    "is_cancellation_code", "negative_quantity", "zero_quantity",
    "nonpositive_price", "missing_customer", "missing_description",
    "invalid_timestamp", "unusual_stock_code", "positive_sale_candidate",
    "customer_sale_candidate",
]
quality = pd.DataFrame({
    "rows": df[flags].sum().astype(int),
    "percent": (df[flags].mean() * 100).round(2),
})
display(quality)

,rows,percent
cross_sheet_duplicate_candidate,45046,4.22
within_sheet_duplicate,12133,1.14
is_cancellation_code,19494,1.83
negative_quantity,22950,2.15
zero_quantity,0,0.0
nonpositive_price,6207,0.58
missing_customer,243007,22.77
missing_description,4382,0.41
invalid_timestamp,0,0.0
unusual_stock_code,10738,1.01


**Finding.** Customer IDs are missing on 243,007 lines (22.77%); descriptions on 4,382 (0.41%). The data contains 19,494 cancellation-coded lines, 22,950 negative-quantity lines, and 6,207 nonpositive-price lines, with no invalid timestamps. The 10,738 unusual-format stock codes are review candidates, not automatically invalid products.

## 5. Cancellations, adjustments, and stock-code anomalies

Inspect negative quantities, cancellation codes, and nonstandard stock codes before defining cleaning rules.


In [5]:
display(pd.crosstab(
    df["is_cancellation_code"],
    df["negative_quantity"],
    rownames=["Cancellation-coded invoice"],
    colnames=["Negative quantity"],
    margins=True,
))

code_review = (
    df.loc[df["unusual_stock_code"]]
      .groupby("stock_code", dropna=False)
      .agg(lines=("StockCode", "size"),
           descriptions=("Description", "nunique"),
           signed_value=("line_amount", "sum"))
      .sort_values("lines", ascending=False)
)
display(code_review.head(25))

display(df.loc[
    df["nonpositive_price"] | df["negative_quantity"],
    ["source_sheet", "invoice_id", "stock_code", "Description",
     "Quantity", "Price", "customer_id", "invoice_timestamp"]
].head(20))

Negative quantity,False,True,All
Cancellation-coded invoice,,,
False,1044420,3457,1047877
True,1,19493,19494
All,1044421,22950,1067371


,lines,descriptions,signed_value
stock_code,,,
POST,2122,1,112341.000
DOT,1446,1,322647.470
M,1421,1,-82796.320
15056BL,923,1,47030.540
C2,282,1,13386.000
79323LP,232,2,5778.390
D,177,1,-13484.540
18098c,142,1,1682.010
79323GR,123,3,2040.860


,source_sheet,invoice_id,stock_code,Description,Quantity,Price,customer_id,invoice_timestamp
178,Year 2009-2010,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.95,16321,2009-12-01 10:33:00
179,Year 2009-2010,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.65,16321,2009-12-01 10:33:00
180,Year 2009-2010,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.25,16321,2009-12-01 10:33:00
181,Year 2009-2010,C489449,21896,POTTING SHED TWINE,-6,2.10,16321,2009-12-01 10:33:00
182,Year 2009-2010,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2.95,16321,2009-12-01 10:33:00
183,Year 2009-2010,C489449,21871,SAVE THE PLANET MUG,-12,1.25,16321,2009-12-01 10:33:00
184,Year 2009-2010,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,1.25,16321,2009-12-01 10:33:00
185,Year 2009-2010,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,0.85,16321,2009-12-01 10:33:00
186,Year 2009-2010,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2.95,16321,2009-12-01 10:33:00
196,Year 2009-2010,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,4.25,17592,2009-12-01 10:44:00


**Finding.** Of 19,494 cancellation-coded lines, 19,493 have negative quantities; 3,457 negative-quantity lines have no cancellation prefix. Special codes such as `POST`, `DOT`, `M`, `D` and `BANK CHARGES` require business-specific classification. Lowercase item-code suffixes also appear among the format flags.

## 6. Candidate sales and validation

Summarize provisional sales subsets and verify basic quality constraints.


In [6]:
# Candidate sets are analytical views, not final cleaned tables.
sales_candidates = df.loc[df["positive_sale_candidate"]].copy()
identified_sales_candidates = df.loc[df["customer_sale_candidate"]].copy()

candidate_summary = pd.Series({
    "Source lines": len(df),
    "Positive sale candidate lines": len(sales_candidates),
    "Customer-linked sale candidate lines": len(identified_sales_candidates),
    "Candidate lines flagged as cross-sheet duplicates": int(
        sales_candidates["cross_sheet_duplicate_candidate"].sum()
    ),
    "Candidate lines with unusual stock codes": int(
        sales_candidates["unusual_stock_code"].sum()
    ),
})
display(candidate_summary.to_frame("count"))

assert df["source_record_key"].is_unique
assert not sales_candidates["is_cancellation_code"].any()
assert sales_candidates["Quantity"].gt(0).all()
assert sales_candidates["Price"].gt(0).all()
assert not sales_candidates["invalid_timestamp"].any()
assert identified_sales_candidates["customer_id"].notna().all()

print("Validation checks passed. No source records were deleted.")

,count
Source lines,1067371
Positive sale candidate lines,1041670
Customer-linked sale candidate lines,805549
Candidate lines flagged as cross-sheet duplicates,44032
Candidate lines with unusual stock codes,9321


Validation checks passed. No source records were deleted.


**Finding.** The provisional positive-sale view contains 1,041,670 lines, including 805,549 customer-linked lines. Within the positive-sale view, 44,032 rows are flagged as cross-sheet duplicate candidates and 9,321 have unusual-format stock codes. All notebook validation assertions passed; these are candidate views, not final deduplicated sales.